# Phase 2.3：qrels、Recall/MRR 与检索交付

## 目标

把“看起来相关”变成可计算评估：为 Query 标注相关 Chunk ID，手算 Recall@k/MRR@k，再调用项目指标函数验证，最后保存 BM25 baseline 报告。

**本课交付：** `data/processed/phase2_evaluation.json`。

## Evidence Quest 任务卡：Phase 2.3：检索裁判席

**你的身份：** 搜索质量裁判  
**案件背景：** 排行榜看起来漂亮不代表真的找到了证据。你要用 qrels 给每个 Query 判分，并找出最值得修复的失败样本。

### 本关专业 Goal

计算 Recall/MRR，形成可解释的检索质量基线。

### 你要交付的作品

**检索挑战赛成绩单 + 失败 Query 案例**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：证据裁判  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. qrels 是什么？

qrels（query relevance judgments）记录“对于某条 Query，哪些文档被认为相关”。它不是检索器自动生成的分数，而是评估标准。

如果没有 qrels，系统只能说“我返回了五条结果”；有了 qrels，才能说“前五条找回了多少相关证据”。小样例适合教学，正式项目还要扩大 Query 并记录标注依据。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase2.3'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase2.3
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 读取 Phase 1 交付的 Chunk 数据。
chunks = json.loads((ROOT / "data" / "processed" / "chunks.json").read_text(encoding="utf-8"))

# 定义两条教学 Query 和它们的关键词。
queries = {"q-001": "Chunk overlap", "q-002": "Dense BM25"}

# 导入生产 BM25 检索器。
from phase2_semantic_search.bm25 import BM25Retriever

# 用真实 Chunk 建立 BM25 索引。
retriever = BM25Retriever(chunks)

# 创建空字典，用于保存每条 Query 的排名 ID。
runs = {}

# 对每条 Query 执行 top-5 检索。
for query_id, query in queries.items():
    # 调用检索器得到带分数的结果。
    results = retriever.search(query, top_k=5)

    # 只保留排名 ID，形成评估函数需要的 run。
    runs[query_id] = [result.doc_id for result in results]

# 打印每条 Query 的排名结果。
print(runs)

{'q-001': ['82ddc7d612e87b02', '3692b05e025373a8'], 'q-002': ['5a62fff245b307a5', '83510b23d2680327']}


## 2. 手工构造教学 qrels

这里用一个透明规则生成小样例标注：正文中包含 `overlap` 的 Chunk 作为 q-001 的相关证据，正文中包含 `Dense` 或 `BM25` 的 Chunk 作为 q-002 的相关证据。真实项目应由人工按照标注规范复核，而不是永远依赖关键词规则。

In [4]:
# 为每条 Query 创建空的相关 ID 集合。
qrels = {"q-001": set(), "q-002": set()}

# 遍历所有 Chunk，按照教学规则挑选相关证据。
for chunk in chunks:
    # 取出正文并统一转为小写，方便英文关键词匹配。
    text = str(chunk["text"]).lower()

    # overlap 出现在正文时，把 Chunk 标记为 q-001 相关。
    if "overlap" in text:
        qrels["q-001"].add(str(chunk["id"]))

    # dense 或 bm25 出现在正文时，把 Chunk 标记为 q-002 相关。
    if "dense" in text or "bm25" in text:
        qrels["q-002"].add(str(chunk["id"]))

# 输出 qrels，观察 Query 和相关 ID 的关系。
print(qrels)

# 两条教学 Query 都应该至少有一条相关 Chunk。
assert all(qrels.values())

{'q-001': {'82ddc7d612e87b02'}, 'q-002': {'5a62fff245b307a5', '83510b23d2680327'}}


## 3. 手算 Recall@k 和 MRR@k

假设排名是 `[wrong, target, other]`，相关集合是 `{target}`：

- Recall@3 = 1/1 = 1，因为唯一相关文档进入前三名。
- MRR@3 = 1/2 = 0.5，因为第一个相关文档排在第 2 名。

两个指标关注不同问题：Recall 关心找没找全，MRR 关心第一个正确结果是否靠前。

In [5]:
# 创建一个用于手算的排名列表。
toy_ranking = ["wrong", "target", "other"]

# 创建手算用的相关文档集合。
toy_relevant = {"target"}

# 截取 top-3 结果并求与相关集合的交集。
retrieved_relevant = set(toy_ranking[:3]) & toy_relevant

# 计算 Recall：召回的相关数量除以全部相关数量。
toy_recall = len(retrieved_relevant) / len(toy_relevant)

# 找到第一个相关结果的排名位置。
first_relevant_rank = toy_ranking.index("target") + 1

# 计算 MRR：第一个相关结果排名的倒数。
toy_mrr = 1 / first_relevant_rank

# 打印手算结果。
print("toy Recall@3:", toy_recall)
print("toy MRR@3:", toy_mrr)

# 验证手算结果。
assert toy_recall == 1.0
assert toy_mrr == 0.5

toy Recall@3: 1.0
toy MRR@3: 0.5


In [6]:
# 从项目模块导入正式的 Recall/MRR 评估函数。
from phase2_semantic_search.metrics import evaluate_qrels, recall_at_k, mrr_at_k

# 计算所有 Query 的平均 Recall@5 和 MRR@5。
metrics = evaluate_qrels(runs, qrels, k=5)

# 打印总体指标。
print(metrics)

# 对每条 Query 单独输出两个指标，定位是哪条 Query 表现不好。
for query_id, relevant_ids in qrels.items():
    # 读取当前 Query 的排名。
    ranked_ids = runs[query_id]

    # 计算当前 Query 的 Recall@5。
    query_recall = recall_at_k(ranked_ids, relevant_ids, k=5)

    # 计算当前 Query 的 MRR@5。
    query_mrr = mrr_at_k(ranked_ids, relevant_ids, k=5)

    # 打印 Query 级别的指标。
    print(query_id, "Recall@5=", query_recall, "MRR@5=", query_mrr)

# 总体指标必须是合法的 0 到 1 之间的数。
assert 0 <= metrics["recall@5"] <= 1
assert 0 <= metrics["mrr@5"] <= 1

{'recall@5': 1.0, 'mrr@5': 1.0}
q-001 Recall@5= 1.0 MRR@5= 1.0
q-002 Recall@5= 1.0 MRR@5= 1.0


## 4. 保存评估证据

报告必须同时保存 Query、qrels、run 和指标。只有一个总分，无法追问某条 Query 为什么失败；只有排名没有 qrels，又无法判断是否正确。

In [7]:
# 把集合转换为排序列表，确保 JSON 输出稳定。
serializable_qrels = {query_id: sorted(relevant_ids) for query_id, relevant_ids in qrels.items()}

# 组合 Phase 2 的评估记录。
evaluation_record = {"queries": queries, "qrels": serializable_qrels, "runs": runs, "metrics": metrics, "k": 5}

# 指定评估记录路径。
evaluation_path = ROOT / "data" / "processed" / "phase2_evaluation.json"

# 写入评估记录，供 Phase 3 读取。
evaluation_path.write_text(json.dumps(evaluation_record, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印交付路径。
print("已生成:", evaluation_path)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\phase2_evaluation.json


## Phase 2 最终闸门

- [ ] 能区分 Query、run 和 qrels。
- [ ] 能手算 Recall 和 MRR，并解释两者不同。
- [ ] 指标来自真实 `chunks.json` 和固定 qrels。
- [ ] 能定位到 Query 级别，而不是只看总分。
- [ ] 已生成 `phase2_evaluation.json`。

## Boss Challenge：新增一道 Query 和 qrels，先预测 Recall/MRR，再运行评估检查预测是否正确。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [8]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [9]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase2_evaluation.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase2_evaluation.json']
